### Importando Pandas e NumPy

In [74]:
import pandas as pd
import numpy as np

### Carregando arquivos

In [75]:
customers = pd.read_csv('./datasets/olist_customers_dataset.csv')
geolocation = pd.read_csv('./datasets/olist_geolocation_dataset.csv')
order_items = pd.read_csv('./datasets/olist_order_items_dataset.csv')
order_payments = pd.read_csv('./datasets/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('./datasets/olist_order_reviews_dataset.csv')
orders = pd.read_csv('./datasets/olist_orders_dataset.csv')
products = pd.read_csv('./datasets/olist_products_dataset.csv')
sellers = pd.read_csv('./datasets/olist_sellers_dataset.csv')
translation = pd.read_csv('./datasets/product_category_name_translation.csv')

### Estrutura das tabelas

In [76]:

tabelas = {
    'customers': customers,
    'orders': orders,
    'order_items': order_items,
    'order_payments': order_payments,
    'order_reviews': order_reviews,
    'geolocation': geolocation,
    'product_category_name_translation': product_category_name_translation,
    'products': products,
    'sellers': sellers
}

for nome, df in tabelas.items():
    print(f'{nome.upper()}\n')
    print(f'COLUNAS: {df.shape[0]}, LINHAS: {df.shape[1]}\n')
    print(f'INFO: \n{df.info()}\n')
    print(f'QUANTIDADE DE NULOS POR COLUNA: \n{df.isnull().sum()}\n')
    print(f'LINHAS DUPLICADAS: \n{df.duplicated().sum()}\n')
    print(f'DESCRICAO: \n{df.describe()}\n')

CUSTOMERS

COLUNAS: 99441, LINHAS: 5

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB
INFO: 
None

QUANTIDADE DE NULOS POR COLUNA: 
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

LINHAS DUPLICADAS: 
0

DESCRICAO: 
       customer_zip_code_prefix
count              99441.000000
mean               35137.474583
std                29797.938996
min                 1003.000000
25%                11347.000000
50%        

### Identificando chaves de relacionamento

In [77]:
print("orders.order_id único?              ", orders["order_id"].is_unique)
print("customers.customer_id único?        ", customers["customer_id"].is_unique)
print("products.product_id único?          ", products["product_id"].is_unique)
print("sellers.seller_id único?          ", sellers["seller_id"].is_unique)
print("order_items order_id + order_item_id é único?", not order_items.duplicated(['order_id', 'order_item_id']).any())
print("order_payments.order_id é único?", order_payments["order_id"].is_unique)
print("order_reviews.order_id é único?", order_reviews["order_id"].is_unique)
print("order_reviews.review_id é único?", order_reviews["review_id"].is_unique)


orders.order_id único?               True
customers.customer_id único?         True
products.product_id único?           True
sellers.seller_id único?           True
order_items order_id + order_item_id é único? True
order_payments.order_id é único? False
order_reviews.order_id é único? False
order_reviews.review_id é único? False


### Tratamento de datas

In [78]:
colunas_orders_data = ["order_purchase_timestamp",     
"order_approved_at",             
"order_delivered_carrier_date",    
"order_delivered_customer_date", 
"order_estimated_delivery_date"
]

for coluna in colunas_orders_data:
    orders[coluna] = pd.to_datetime(orders[coluna])
    
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

order_reviews["review_creation_date"] = pd.to_datetime(order_reviews["review_creation_date"])
order_reviews["review_answer_timestamp"] = pd.to_datetime(order_reviews["review_answer_timestamp"])

### Verifica pedidos marcados como entregue, mas sem data de entrega

In [79]:
delivered = orders["order_status"] == "delivered"
print("delivered sem data de entrega:", (delivered & orders["order_delivered_customer_date"].isna()).sum())

delivered sem data de entrega: 8


### Pedido com mais de um review. Manter o mais mais recente apenas

In [80]:
reviews_recentes = (order_reviews.sort_values("review_answer_timestamp").drop_duplicates("order_id", keep="last"))
print(len(order_reviews), "->", len(reviews_recentes), "reviews | pedidos únicos:", reviews_recentes["order_id"].is_unique)

99224 -> 98673 reviews | pedidos únicos: True


### Agregando itens e pagamentos por pedido

In [81]:
items_agg = (order_items.groupby("order_id")
             .agg(n_itens=("order_item_id", "count"),
                  n_vendedores=("seller_id", "nunique"),
                  valor_produtos=("price", "sum"),
                  valor_frete=("freight_value", "sum"))
             .reset_index())

pay_agg = (order_payments.groupby("order_id")
           .agg(valor_pago=("payment_value", "sum"),
                n_pagamentos=("payment_sequential", "count"),
                parcelas_max=("payment_installments", "max"))
           .reset_index())

pay_main = (order_payments.sort_values("payment_value", ascending=False)
                    .drop_duplicates("order_id")[["order_id", "payment_type"]]
                    .rename(columns={"payment_type": "tipo_pagamento"}))

### Base de pedidos

In [82]:
pedidos = (orders
    .merge(customers, on="customer_id", how="left", validate="m:1")
    .merge(items_agg, on="order_id", how="left", validate="1:1")
    .merge(pay_agg, on="order_id", how="left", validate="1:1")
    .merge(pay_main, on="order_id", how="left", validate="1:1")
    .merge(reviews_recentes[["order_id", "review_score"]], on="order_id", how="left", validate="1:1"))


In [83]:
pedidos["mes_compra"] = pedidos["order_purchase_timestamp"].dt.to_period("M")
pedidos["tempo_entrega_dias"] = (pedidos["order_delivered_customer_date"] - pedidos["order_purchase_timestamp"]).dt.total_seconds() / 86400
pedidos["atraso_dias"] = (pedidos["order_delivered_customer_date"] - pedidos["order_estimated_delivery_date"]).dt.total_seconds() / 86400
pedidos["atrasado"] = (pedidos["atraso_dias"] > 0).astype("boolean").where(pedidos["atraso_dias"].notna())

In [84]:
regiao = {**dict.fromkeys(["SP", "RJ", "MG", "ES"], "Sudeste"),
          **dict.fromkeys(["PR", "SC", "RS"], "Sul"),
          **dict.fromkeys(["DF", "GO", "MT", "MS"], "Centro-Oeste"),
          **dict.fromkeys(["BA", "SE", "AL", "PE", "PB", "RN", "CE", "PI", "MA"], "Nordeste"),
          **dict.fromkeys(["AM", "PA", "AC", "RO", "RR", "AP", "TO"], "Norte")}
pedidos["regiao"] = pedidos["customer_state"].map(regiao)

### Subconjuntos usados nas métricas. Cancelados e indisponíveis não geram receita. Entrega só faz sentido onde ela ocorreu

In [85]:
validos = pedidos[~pedidos["order_status"].isin(["canceled", "unavailable"]) & pedidos["valor_pago"].notna()]
entregues = pedidos[(pedidos["order_status"] == "delivered") & pedidos["order_delivered_customer_date"].notna()]
print(f"Pedidos totais: {len(pedidos):,} | válidos (receita): {len(validos):,} | entregues: {len(entregues):,}")

Pedidos totais: 99,441 | válidos (receita): 98,206 | entregues: 96,470


### Base de itens, uma linha por item

In [86]:
products = products.merge(translation, on="product_category_name", how="left", validate="m:1")

products["categoria"] = (products["product_category_name_english"].fillna(products["product_category_name"]).fillna("sem_categoria"))

itens = (order_items
    .merge(products[["product_id", "categoria", "product_weight_g"]], on="product_id", how="left", validate="m:1")
    .merge(sellers[["seller_id", "seller_state", "seller_city"]], on="seller_id", how="left", validate="m:1")
    .merge(pedidos[["order_id", "order_status", "mes_compra", "customer_state", "regiao",
                    "review_score", "atrasado", "tempo_entrega_dias"]], on="order_id", how="left", validate="m:1"))

itens_validos = itens[~itens["order_status"].isin(["canceled", "unavailable"])]
itens_validos.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,categoria,product_weight_g,seller_state,seller_city,order_status,mes_compra,customer_state,regiao,review_score,atrasado,tempo_entrega_dias
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,650.0,SP,volta redonda,delivered,2017-09,RJ,Sudeste,5.0,False,7.614421
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,30000.0,SP,sao paulo,delivered,2017-04,SP,Sudeste,4.0,False,16.216181
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,furniture_decor,3050.0,MG,borda da mata,delivered,2018-01,MG,Sudeste,5.0,False,7.948437
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumery,200.0,SP,franca,delivered,2018-08,SP,Sudeste,4.0,False,6.147269
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,garden_tools,3750.0,PR,loanda,delivered,2017-02,SP,Sudeste,5.0,False,25.114352


## Métricas de negócio

### Quantidade de pedidos, ticket médio e frete médio

Definições adotadas:
- Ticket médio = valor pago médio por pedido válido (`payment_value`, inclui frete). Também mostramos o ticket só de produtos.
- Frete médio = frete médio por pedido válido.

In [87]:
print(f"Total de pedidos (todos os status):   {len(pedidos):,}")
print(f"Pedidos válidos (sem cancel/indisp.):  {len(validos):,}")
print(f"Clientes únicos (customer_unique_id):  {pedidos['customer_unique_id'].nunique():,}")
print(f"Ticket médio (valor pago):             R$ {validos['valor_pago'].mean():,.2f}")
print(f"Ticket mediano:                        R$ {validos['valor_pago'].median():,.2f}")
print(f"Ticket médio (só produtos):            R$ {validos['valor_produtos'].mean():,.2f}")
print(f"Frete médio por pedido:                R$ {validos['valor_frete'].mean():,.2f}")
print(f"Frete como % do valor dos produtos:    {validos['valor_frete'].sum() / validos['valor_produtos'].sum():.1%}")
print()
print(pedidos["order_status"].value_counts(normalize=True).map("{:.2%}".format))

Total de pedidos (todos os status):   99,441
Pedidos válidos (sem cancel/indisp.):  98,206
Clientes únicos (customer_unique_id):  96,096
Ticket médio (valor pago):             R$ 160.27
Ticket mediano:                        R$ 105.28
Ticket médio (só produtos):            R$ 137.42
Frete médio por pedido:                R$ 22.82
Frete como % do valor dos produtos:    16.6%

order_status
delivered      97.02%
shipped         1.11%
canceled        0.63%
unavailable     0.61%
invoiced        0.32%
processing      0.30%
created         0.01%
approved        0.00%
Name: proportion, dtype: str


### Faturamento por período

Faturamento = soma de valor_pago dos pedidos válidos. 

In [90]:
fat_mensal = (validos.groupby("mes_compra").agg(pedidos=("order_id", "count"), faturamento=("valor_pago", "sum"), ticket_medio=("valor_pago", "mean")))
fat_mensal["crescimento_mom"] = fat_mensal["faturamento"].pct_change()
display(fat_mensal)


,pedidos,faturamento,ticket_medio,crescimento_mom
mes_compra,,,,
2016-09,1,136.23,136.230000,NaN
2016-10,293,51657.53,176.305563,378.193496
2016-12,1,19.62,19.620000,-0.999620
2017-01,787,137006.76,174.087370,6982.015291
2017-02,1718,283621.94,165.088440,1.070131
2017-03,2617,425656.40,162.650516,0.500788
2017-04,2377,405988.38,170.798645,-0.046206
2017-05,3640,582926.16,160.144549,0.435820
2017-06,3205,499827.47,155.952409,-0.142554


In [91]:
validos = validos.assign(dia_semana=validos["order_purchase_timestamp"].dt.day_name(), hora=validos["order_purchase_timestamp"].dt.hour)

ordem = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

por_dia = validos["dia_semana"].value_counts().reindex(ordem)
por_hora = validos["hora"].value_counts().sort_index()

display(por_dia.to_frame("pedidos").assign(pct=(por_dia / por_dia.sum()).map("{:.1%}".format)))
display(por_hora.to_frame("pedidos").assign(pct=(por_hora / por_hora.sum()).map("{:.1%}".format)))

,pedidos,pct
dia_semana,,
Monday,15979,16.3%
Tuesday,15760,16.0%
Wednesday,15353,15.6%
Thursday,14570,14.8%
Friday,13949,14.2%
Saturday,10764,11.0%
Sunday,11831,12.0%


,pedidos,pct
hora,,
0,2362,2.4%
1,1154,1.2%
2,503,0.5%
3,267,0.3%
4,205,0.2%
5,185,0.2%
6,493,0.5%
7,1218,1.2%
8,2939,3.0%


### Entrega: tempo médio e atrasos

- Tempo de entrega = data de entrega ao cliente − data da compra (só pedidos entregues).
- Atraso = entrega real > data estimada.

In [96]:
print(f"Tempo médio de entrega:   {entregues['tempo_entrega_dias'].mean():.1f} dias")
print(f"Tempo mediano de entrega: {entregues['tempo_entrega_dias'].median():.1f} dias")
print(f"Estimativa média (compra -> prazo prometido):, "f"{((entregues['order_estimated_delivery_date'] - entregues['order_purchase_timestamp']).dt.total_seconds()/86400).mean():.1f} dias")
print(f"Pedidos atrasados: {entregues['atrasado'].sum():,.0f} ({entregues['atrasado'].mean():.1%})")
print(f"Atraso médio entre os atrasados: {entregues.loc[entregues['atrasado'] == True, 'atraso_dias'].mean():.1f} dias")
print(f"Antecedência média entre os no prazo: {-entregues.loc[entregues['atrasado'] == False, 'atraso_dias'].mean():.1f} dias")

atraso_mensal = entregues.groupby("mes_compra").agg(entregas=("order_id", "count"), pct_atrasado=("atrasado", "mean"), tempo_medio=("tempo_entrega_dias", "mean"))
display(atraso_mensal.tail(24))

Tempo médio de entrega:   12.6 dias
Tempo mediano de entrega: 10.2 dias
Estimativa média (compra -> prazo prometido):, 23.7 dias
Pedidos atrasados: 7,826 (8.1%)
Atraso médio entre os atrasados: 9.6 dias
Antecedência média entre os no prazo: 13.0 dias


,entregas,pct_atrasado,tempo_medio
mes_compra,,,
2016-09,1,1.0,54.813194
2016-10,265,0.011321,19.600559
2016-12,1,0.0,4.693021
2017-01,750,0.030667,12.647044
2017-02,1653,0.032063,13.168825
2017-03,2546,0.055774,12.951184
2017-04,2303,0.078593,14.917913
2017-05,3545,0.036107,11.322363
2017-06,3135,0.038596,12.011573


### Categorias e produtos mais vendidos

"Vendido" = quantidade de itens (cada linha de order_items é uma unidade). Também mostramos por receita (price), pois os rankings podem diferir.
O dataset não traz nome de produto, apenas product_id.

In [95]:
cat = (itens_validos.groupby("categoria").agg(itens_vendidos=("order_item_id", "count"),
            pedidos=("order_id", "nunique"),
            receita=("price", "sum"),
            preco_medio=("price", "mean"),
            frete_medio=("freight_value", "mean"),
            nota_media=("review_score", "mean"),
            pct_atrasado=("atrasado", "mean"),
            tempo_entrega=("tempo_entrega_dias", "mean")).sort_values("itens_vendidos", ascending=False))

cat["pct_receita"] = cat["receita"] / cat["receita"].sum()
cat["frete_sobre_preco"] = cat["frete_medio"] / cat["preco_medio"]
display(cat.head(15))

,itens_vendidos,pedidos,receita,preco_medio,frete_medio,nota_media,pct_atrasado,tempo_entrega,pct_receita,frete_sobre_preco
categoria,,,,,,,,,,
bed_bath_table,11097,9399,1035964.06,93.355327,18.421381,3.901313,0.083995,12.806463,0.076770,0.197325
health_beauty,9634,8800,1255695.13,130.339955,18.888380,4.149707,0.090544,11.979077,0.093053,0.144916
sports_leisure,8590,7673,979740.92,114.055986,19.499850,4.122260,0.07414,12.149278,0.072604,0.170967
furniture_decor,8298,6425,727465.05,87.667516,20.747287,3.918807,0.084314,12.880255,0.053909,0.236659
computers_accessories,7781,6654,904322.02,116.221825,18.843280,3.947402,0.077718,13.225304,0.067015,0.162132
housewares,6915,5847,626825.80,90.647260,20.999181,4.071169,0.064901,10.922552,0.046451,0.231658
watches_gifts,5970,5604,1198185.21,200.701040,16.768385,4.027703,0.082807,12.663180,0.088791,0.083549
telephony,4527,4183,322342.64,71.204471,15.675586,3.953737,0.083296,12.861938,0.023887,0.220149
garden_tools,4328,3505,481009.94,111.139080,22.762306,4.056099,0.079663,13.708882,0.035645,0.204809


### Comparação regional

Comparamos por estado e região do cliente: volume, ticket, frete, tempo de entrega, atrasos e nota.

In [97]:
def resumo_regional(chave):
    base = validos.groupby(chave).agg(pedidos=("order_id", "count"),faturamento=("valor_pago", "sum"), ticket_medio=("valor_pago", "mean"), frete_medio=("valor_frete", "mean"))
    ent = entregues.groupby(chave).agg(tempo_entrega=("tempo_entrega_dias", "mean"), pct_atrasado=("atrasado", "mean"), nota_media=("review_score", "mean"))
    out = base.join(ent)
    out["pct_pedidos"] = out["pedidos"] / out["pedidos"].sum()
    out["frete_sobre_produtos"] = (validos.groupby(chave)["valor_frete"].sum() / validos.groupby(chave)["valor_produtos"].sum())
    return out.sort_values("pedidos", ascending=False)

reg = resumo_regional("regiao")
uf = resumo_regional("customer_state")
display(reg)
display(uf)

,pedidos,faturamento,ticket_medio,frete_medio,tempo_entrega,pct_atrasado,nota_media,pct_pedidos,frete_sobre_produtos
regiao,,,,,,,,,
Sudeste,67338,10161496.39,150.902854,19.862117,10.751910,0.074509,4.179520,0.685681,0.151593
Sul,14000,2280637.51,162.902679,24.399098,14.032353,0.070513,4.191270,0.142557,0.176205
Nordeste,9305,1872976.48,201.287102,35.934877,20.005013,0.143299,3.970251,0.094750,0.217379
Centro-Oeste,5729,1014265.96,177.040663,26.523619,15.017779,0.079659,4.128531,0.058337,0.176286
Norte,1834,409760.67,223.424575,41.390992,22.592321,0.097996,4.031478,0.018675,0.227405


,pedidos,faturamento,ticket_medio,frete_medio,tempo_entrega,pct_atrasado,nota_media,pct_pedidos,frete_sobre_produtos
customer_state,,,,,,,,,
SP,41126,5878935.99,142.949375,17.368358,8.761386,0.058947,4.245790,0.418773,0.138322
RJ,12698,2115994.28,166.639965,23.946140,15.309438,0.134737,3.965441,0.129300,0.167830
MG,11496,1843460.37,160.356678,23.448698,12.008666,0.056104,4.192291,0.117060,0.171315
RS,5417,877803.80,162.046114,24.881036,15.300276,0.071482,4.184754,0.055160,0.181441
PR,4983,794653.20,159.472848,23.547481,11.991582,0.04997,4.239796,0.050740,0.173314
SC,3600,608180.51,168.939031,24.852854,14.954783,0.097575,4.133561,0.036658,0.172482
BA,3344,607041.45,181.531534,29.844447,19.335466,0.140356,3.930629,0.034051,0.196802
DF,2121,351487.63,165.717883,23.792811,12.967568,0.070673,4.131401,0.021597,0.167641
ES,2018,323105.75,160.111868,24.553469,15.789307,0.122306,4.078720,0.020549,0.181145


### Avaliações: média, distribuição e relação com atraso

In [101]:
nota = pedidos["review_score"].dropna()
print(f"Nota média: {nota.mean():.2f} | mediana: {nota.median():.0f} | pedidos avaliados: {len(nota):,} ({len(nota)/len(pedidos):.1%})")
dist = nota.value_counts(normalize=True).sort_index()
display(dist.map("{:.1%}".format).to_frame("% das avaliações"))

e = entregues.dropna(subset=["review_score"])
print(e.groupby("atrasado")["review_score"].agg(["count", "mean"]).rename(index={False: "no prazo", True: "atrasado"}))
print("\nDistribuição das notas (%) no prazo vs. atrasado:")
display(pd.crosstab(e["atrasado"].map({False: "no prazo", True: "atrasado"}), e["review_score"], normalize="index").mul(100).round(1))

Nota média: 4.09 | mediana: 5 | pedidos avaliados: 98,673 (99.2%)


,% das avaliações
review_score,
1.0,11.5%
2.0,3.2%
3.0,8.2%
4.0,19.3%
5.0,57.8%


          count      mean
atrasado                 
no prazo  88163  4.294114
atrasado   7661  2.565070

Distribuição das notas (%) no prazo vs. atrasado:


review_score,1.0,2.0,3.0,4.0,5.0
atrasado,,,,,
atrasado,46.2,7.9,11.4,12.3,22.2
no prazo,6.6,2.6,8.0,20.4,62.4


## Observações

### Observação 1: pedidos atrasados recebem notas muito piores

Evidência: na base entregues :
- Nota média dos pedidos no prazo: 4,29. Nota média dos atrasados: 2,57.
- Entre os atrasados, 46,2% deram nota 1, contra 6,6% entre os pontuais.
- Entre os pontuais, 62,4% deram nota 5, contra 22,2% entre os atrasados.
- Para comparação, a base geral tem nota média de 4,09 e mediana 5, com 57,8% das notas iguais a 5.

Hipótese: o atraso é um dos principais fatores de insatisfação. O cliente avalia a entrega e não só o produto: quando o pedido chega depois da data estimada, a expectativa é quebrada e a nota cai para o extremo. Isso explicaria por que a distribuição dos atrasados é quase "oposta" à dos pontuais: concentrada na nota 1, enquanto a dos pontuais se concentra na nota 5.

Ressalva: isso é correlação. Pedidos atrasados podem ter outras características em comum (produtos mais complexos, vendedores piores, regiões mais distantes), e esses fatores também poderiam baixar a nota.

Aprofundamento:
- Verificar se a nota piora com a gravidade do atraso (1 dia, 5 dias, 15 dias) ou se qualquer atraso já derruba a nota.
- Separar por região, para ver se o efeito do atraso é o mesmo em regios diferentes.

### Observação 2: quanto mais longe do Sudeste, maior o frete e mais lenta a entrega, e o Nordeste é o mais penalizado

**Evidência:** (base validos para frete; base entregues para prazo, atraso e nota; agrupado pela região do cliente)
- Frete médio por pedido: Sudeste R$ 19,86, Sul R$ 24,40, Centro-Oeste R$ 26,52, Nordeste R$ 35,93 e Norte R$ 41,39.
- Tempo médio de entrega: Sudeste 10,8 dias, Nordeste 20,0 dias e Norte 22,6 dias.
- O Nordeste tem o maior percentual de atrasos entre as regiões (14,3%, contra 7,5% no Sudeste) e a menor nota média (3,97, contra 4,18 no Sudeste).
- Entre os estados, o padrão se repete: Sao Paulo tem frete de R$ 17,37, entrega em 8,8 dias e 5,9% de atrasos, enquanto Maranhao tem frete de R$ 42,66, entrega em 21,6 dias, 19,7% de atrasos e nota de 3,83. Alagoas tem 23,9% de atrasos e 24,5 dias de entrega.

**Hipótese:** a distância entre vendedor e cliente é a principal explicação. Se a maior parte dos vendedores está no Sudeste, pedidos para o Norte e o Nordeste percorrem trajetos longos, o que encarece o frete, alonga o prazo e aumenta a chance de atraso.

**Ressalvas:**
- Estados pequenos têm poucos pedidos (AL: 411, SE: 345, AP: 68, RR: 45), então suas porcentagens oscilam bastante
- Nem tudo segue o padrão da distância: o Norte tem 9,8% de atrasos, menos que o Nordeste, e AM (4,1%) e RO (2,9%) têm poucos atrasos apesar de entregas longas.

**Aprofundamento:**
- Cruzar seller_state com customer_state para ver se a distância realmente explica o frete e o prazo, e se clientes do Nordeste atendidos por vendedores locais têm desempenho melhor.
- Comparar o prazo prometido por região com o prazo real.
- Verificar se a nota do Nordeste continua menor quando se compara apenas pedidos no prazo.

### Observação 3: as categorias mais vendidas em unidades não são as que mais geram receita, e algumas têm nota baixa que o atraso não explica

**Evidência:** base itens_validos:
- bed_bath_table lidera em itens (11.097), mas está em 3º em receita (R$ 1,04 mi). health_beauty é 2ª em itens (9.634) e 1ª em receita (R$ 1,26 mi).
- watches_gifts é a 7ª em itens (5.970), mas a 2ª em receita (R$ 1,20 mi, 8,9% do total), porque seu preço médio é R$ 200,70, contra R$ 93,36 em bed_bath_table e R$ 71,20 em telephony.
- No extremo oposto, electronics vende 2.755 itens, mas gera só R$ 157 mil.
- As notas médias também diferem: bed_bath_table (3,90), furniture_decor (3,92) e computers_accessories (3,95) têm as menores entre as 15 maiores, enquanto health_beauty (4,15) e toys (4,18) têm as maiores. O atraso não separa esses grupos: bed_bath_table tem 8,4% de atrasos e health_beauty tem 9,1%, mas notas bem diferentes.

**Hipótese:** o ranking por unidades mede popularidade e o ranking por receita mede valor. Categorias de preço alto e poucas unidades, como watches_gifts, podem ser mais lucrativas que categorias de alto volume e preço baixo. O frete é fixo em parte, então pesa mais em itens baratos, e isso explicaria o frete alto sobre o preço em electronics e furniture_decor. Sobre as notas, categorias como bed_bath_table e furniture_decor podem gerar mais frustração por diferença entre o produto esperado e o recebido.

**Ressalvas:**
- A tabela mostra só as 15 categorias mais vendidas em itens. Para afirmar o ranking de receita, é preciso ordenar a tabela completa por receita.
- nota_media, pct_atrasado e tempo_entrega são do pedido e se repetem em cada item, então pedidos com muitos itens pesam mais na média.
- Diferenças pequenas de nota (3,90 vs. 4,15) podem ter explicações como mistura de produtos dentro da categoria.

**Aprofundamento:**
- Ordenar todas as categorias por receita e comparar com o ranking por itens, filtrando categorias com mais de 100 itens.
- Verificar se a nota dos itens de bed_bath_table continua menor quando se olham só pedidos no prazo.
- Testar a relação entre peso do produto e frete para ver se explica o frete_sobre_preco alto em categorias como furniture_decor.

## Resumo das decisões de dados

 geolocation não utilizada  várias linhas por CEP e cliente/vendedor já têm cidade e UF; não é necessária para as métricas 

 Reviews: manter a mais recente por pedido  alguns pedidos têm mais de uma avaliação, o que duplicaria pedidos no join 

 Itens, pagamentos e reviews agregados antes do join  evitar multiplicação de linhas 

 Receita exclui canceled e unavailable  não geraram faturamento efetivo 
 
 Métricas de entrega só com delivered e data de entrega preenchida  sem data não há como medir prazo 